# ✈️ Prévision du Nombre de Passagers Aériens (RNN vs LSTM vs GRU)

## Présentation du projet

Dans ce notebook, nous allons prédire le **nombre mensuel de passagers aériens internationaux** à partir de l'historique des mois précédents, en comparant **trois architectures récurrentes**.

### 🎯 Ce que vous allez apprendre :
- Prétraiter des données de séries temporelles pour des RNN
- Construire et entraîner **SimpleRNN, LSTM et GRU**
- Comparer leurs performances avec des métriques et des graphiques
- Visualiser l'historique d'entraînement et les prédictions

### 📊 Le Dataset
Le classique **Airline Passengers** : nombre de passagers (en milliers) chaque mois de 1949 à 1960.

---
> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

---
## 📦 PARTIE 1 — Préparation des données

**Objectif :** Charger, explorer, nettoyer et préparer les données pour l'entraînement de RNN.

In [ ]:
# ── Installation des bibliothèques ────────────────────────────────────────────
!pip install -q tensorflow pandas matplotlib seaborn scikit-learn

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

sns.set_theme(style='darkgrid')
%matplotlib inline

print(f"✅ Bibliothèques importées — TensorFlow {tf.__version__}")
print(f"🖥️  GPU disponible : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Téléchargement du dataset ─────────────────────────────────────────────────
url = ("https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/"
       "Week%206/W6D4/Airline%20Passenger%20Forecasting%20by%20RNN,%20LSTM,%20&%20GRU.zip")
zip_path = "airline_data.zip"

print("📥 Téléchargement du dataset...")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('./airline_data')

print("✅ Dataset téléchargé !")
for root, dirs, files in os.walk('./airline_data'):
    for f in files:
        print("  ", os.path.join(root, f))

In [ ]:
# ── Chargement du dataset ─────────────────────────────────────────────────────
# Trouver le fichier CSV (le nom peut varier légèrement selon l'archive)
csv_file = None
for root, dirs, files in os.walk('./airline_data'):
    for f in files:
        if f.endswith('.csv'):
            csv_file = os.path.join(root, f)
            break

print(f"📂 Fichier trouvé : {csv_file}")
df = pd.read_csv(csv_file)

print("\n📋 Aperçu des données :")
df.head()

In [ ]:
# ── Exploration de la structure ───────────────────────────────────────────────
print(f"📐 Forme du dataset : {df.shape}")
print(f"\n🔍 Types de données :")
print(df.dtypes)
print(f"\n📊 Statistiques descriptives :")
print(df.describe())
print(f"\n❓ Valeurs manquantes :")
print(df.isnull().sum())

In [ ]:
# ── Nettoyage : suppression des valeurs manquantes ────────────────────────────
n_before = len(df)
df = df.dropna()
n_after = len(df)

print(f"🧹 Lignes avant nettoyage : {n_before}")
print(f"🧹 Lignes après nettoyage : {n_after}")
print(f"   ({n_before - n_after} lignes supprimées)")

# Identifier la colonne des passagers (généralement la 2ème colonne)
# On affiche les noms de colonnes pour être sûr
print(f"\n📋 Colonnes disponibles : {list(df.columns)}")

# La colonne cible est généralement nommée 'Passengers' ou similaire
passenger_col = [c for c in df.columns if 'passenger' in c.lower()]
passenger_col = passenger_col[0] if passenger_col else df.columns[-1]
print(f"   Colonne cible identifiée : '{passenger_col}'")

In [ ]:
# ── Visualisation de la série temporelle brute ───────────────────────────────
passengers = df[passenger_col].values.astype(float)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(passengers, color='royalblue', linewidth=1.5, marker='o', markersize=3)
ax.set_title('Nombre de passagers aériens internationaux (mensuel)', fontsize=13)
ax.set_xlabel('Mois (depuis le début des données)')
ax.set_ylabel('Passagers (milliers)')
plt.tight_layout()
plt.show()

print("💡 On observe une tendance croissante ET une saisonnalité annuelle marquée.")
print(f"   Nombre total de mois : {len(passengers)}")

In [ ]:
# ── Normalisation avec MinMaxScaler ──────────────────────────────────────────
# Les RNN convergent beaucoup mieux avec des données normalisées (0 à 1)
# car les fonctions d'activation (tanh, sigmoid) saturent pour les grandes valeurs.

scaler = MinMaxScaler(feature_range=(0, 1))
passengers_scaled = scaler.fit_transform(passengers.reshape(-1, 1))

print(f"✅ Données normalisées :")
print(f"   Avant : min={passengers.min():.0f}, max={passengers.max():.0f}")
print(f"   Après : min={passengers_scaled.min():.2f}, max={passengers_scaled.max():.2f}")

In [ ]:
# ── Création des séquences X (prédicteurs) et y (cible) ──────────────────────
# On utilise une fenêtre de 6 mois passés pour prédire le mois suivant

LOOK_BACK = 6  # Nombre de mois passés utilisés pour la prédiction

def create_sequences(data, look_back):
    """
    Transforme une série temporelle en paires (X, y) pour l'entraînement supervisé.

    Args:
        data      : série temporelle normalisée, shape (n_mois, 1)
        look_back : nombre de mois passés à utiliser comme features

    Returns:
        X : shape (n_exemples, look_back, 1)
        y : shape (n_exemples,)
    """
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i : i + look_back, 0])   # 6 valeurs passées
        y.append(data[i + look_back, 0])        # valeur du mois suivant
    return np.array(X), np.array(y)

X, y = create_sequences(passengers_scaled, LOOK_BACK)

# Reshape pour le format attendu par les couches RNN : (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"📐 Formes après création des séquences :")
print(f"   X : {X.shape}  → (exemples, {LOOK_BACK} mois passés, 1 feature)")
print(f"   y : {y.shape}")

In [ ]:
# ── Découpage Train / Test ───────────────────────────────────────────────────
# IMPORTANT : pas de mélange (shuffle) — on respecte l'ordre chronologique

TRAIN_RATIO = 0.80
split_idx   = int(len(X) * TRAIN_RATIO)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"🔀 Découpage chronologique :")
print(f"   Train : {len(X_train)} exemples")
print(f"   Test  : {len(X_test)} exemples")

# Visualisation du split
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(passengers)), passengers, color='lightgray', linewidth=1, label='Série complète')
split_point = split_idx + LOOK_BACK
ax.axvline(split_point, color='red', linestyle='--', linewidth=2, label='Frontière Train/Test')
ax.fill_betweenx([passengers.min(), passengers.max()], 0, split_point, alpha=0.1, color='royalblue', label='Train')
ax.fill_betweenx([passengers.min(), passengers.max()], split_point, len(passengers), alpha=0.1, color='tomato', label='Test')
ax.set_title('Découpage Train / Test (chronologique)', fontsize=13)
ax.set_xlabel('Mois'); ax.set_ylabel('Passagers')
ax.legend()
plt.tight_layout()
plt.show()

---
## 🧠 PARTIE 2 — Construction des modèles

On construit **3 architectures** avec une structure similaire pour une comparaison équitable :
```
Input (6 mois) → Couche récurrente (×2, empilées) → Dropout → Dense(1)
```

| Modèle | Couche Keras | Particularité |
|--------|-------------|---------------|
| SimpleRNN | `SimpleRNN` | Le plus simple, sensible au vanishing gradient |
| LSTM | `LSTM` | Cellule mémoire + 3 portes, bonne mémoire long-terme |
| GRU | `GRU` | 2 portes, plus léger que LSTM, performances similaires |

In [ ]:
# ── Modèle 1 : Simple RNN ─────────────────────────────────────────────────────
def build_simple_rnn(look_back, units=50):
    model = Sequential([
        # Première couche RNN : return_sequences=True pour empiler une 2ème couche
        SimpleRNN(units, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        # Deuxième couche RNN : return_sequences=False (dernière couche récurrente)
        SimpleRNN(units, return_sequences=False),
        Dropout(0.2),
        # Couche de sortie : 1 valeur (prédiction du mois suivant)
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

rnn_model = build_simple_rnn(LOOK_BACK)
print("📐 Architecture SimpleRNN :")
rnn_model.summary()

In [ ]:
# ── Modèle 2 : LSTM ───────────────────────────────────────────────────────────
def build_lstm(look_back, units=50):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        LSTM(units, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

lstm_model = build_lstm(LOOK_BACK)
print("📐 Architecture LSTM :")
lstm_model.summary()

In [ ]:
# ── Modèle 3 : GRU ────────────────────────────────────────────────────────────
def build_gru(look_back, units=50):
    model = Sequential([
        GRU(units, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        GRU(units, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

gru_model = build_gru(LOOK_BACK)
print("📐 Architecture GRU :")
gru_model.summary()

In [ ]:
# ── Comparaison du nombre de paramètres ──────────────────────────────────────
models_dict = {'SimpleRNN': rnn_model, 'LSTM': lstm_model, 'GRU': gru_model}

print("📊 Nombre de paramètres par modèle :\n")
for name, model in models_dict.items():
    n_params = model.count_params()
    print(f"   {name:<10} : {n_params:,} paramètres")

print("\n💡 LSTM a le plus de paramètres (4 portes), GRU est intermédiaire (3 portes),")
print("   SimpleRNN en a le moins (pas de portes, juste un état caché simple).")

---
## 🚀 PARTIE 3 — Entraînement des modèles

On entraîne les 3 modèles avec **Early Stopping** pour éviter le surapprentissage.

In [ ]:
# ── Configuration et entraînement ────────────────────────────────────────────
EPOCHS     = 200
BATCH_SIZE = 8

def make_early_stop():
    """Crée un nouveau callback EarlyStopping (un par modèle)."""
    return EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True
    )

histories = {}

for name, model in models_dict.items():
    print(f"\n🏋️  Entraînement de {name}...")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.15,
        callbacks=[make_early_stop()],
        verbose=0  # Silencieux pour ne pas encombrer le notebook
    )
    histories[name] = history
    n_epochs_run = len(history.history['loss'])
    final_loss   = history.history['val_loss'][-1]
    print(f"   ✅ Arrêté après {n_epochs_run} epochs | Val Loss finale : {final_loss:.5f}")

---
## 📈 PARTIE 4 — Visualisation de l'historique d'entraînement

On compare les courbes de loss (train et validation) des 3 modèles.

In [ ]:
# ── Graphiques individuels : Train vs Validation Loss ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = {'SimpleRNN': 'tomato', 'LSTM': 'royalblue', 'GRU': 'seagreen'}

for ax, (name, history) in zip(axes, histories.items()):
    ax.plot(history.history['loss'],     label='Train Loss',      color=colors[name], linewidth=2)
    ax.plot(history.history['val_loss'], label='Validation Loss', color=colors[name], linewidth=2, linestyle='--', alpha=0.7)
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend()

plt.suptitle('Historique d\'entraînement par modèle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparaison directe des 3 modèles (validation loss) ─────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

for name, history in histories.items():
    ax.plot(history.history['val_loss'], label=f'{name} (val)', color=colors[name], linewidth=2)

ax.set_title('Comparaison des courbes de Validation Loss', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.show()

print("💡 Observations possibles :")
print("   - Un modèle qui converge plus vite n'est pas toujours le meilleur en généralisation")
print("   - Surveillez les écarts entre train et validation loss (signe d'overfitting)")

---
## 🔮 PARTIE 5 — Prédictions sur le jeu de test

On utilise les modèles entraînés pour prédire les valeurs de test, puis on **inverse la normalisation** pour revenir à l'échelle réelle (nombre de passagers).

In [ ]:
# ── Prédictions et dénormalisation ───────────────────────────────────────────
predictions = {}

for name, model in models_dict.items():
    # Prédiction sur le jeu de test (valeurs normalisées 0-1)
    y_pred_scaled = model.predict(X_test, verbose=0)

    # Inverser la normalisation pour revenir à l'échelle réelle (passagers)
    y_pred_real = scaler.inverse_transform(y_pred_scaled)

    predictions[name] = y_pred_real.flatten()
    print(f"✅ Prédictions calculées pour {name}")

# Dénormaliser également les vraies valeurs de test (pour comparaison)
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

print(f"\n📐 Forme des prédictions : {predictions['SimpleRNN'].shape}")
print(f"📐 Forme des valeurs réelles : {y_test_real.shape}")

In [ ]:
# ── Calcul des métriques de performance ──────────────────────────────────────
metrics_results = {}

print("📊 Métriques de performance (échelle réelle, en milliers de passagers) :\n")
for name, y_pred in predictions.items():
    mse  = mean_squared_error(y_test_real, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test_real, y_pred)
    r2   = r2_score(y_test_real, y_pred)

    metrics_results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"  [{name:<10}] RMSE: {rmse:>7.2f} | MAE: {mae:>7.2f} | R²: {r2:.4f}")

---
## 📊 PARTIE 6 — Visualisation des résultats de prédiction

On compare visuellement les prédictions de chaque modèle aux vraies valeurs, puis on détermine **le meilleur modèle**.

In [ ]:
# ── Graphiques individuels : Réel vs Prédit pour chaque modèle ───────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)

for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.plot(y_test_real, label='Valeurs réelles',  color='black',       linewidth=2, marker='o', markersize=4)
    ax.plot(y_pred,       label=f'Prédiction {name}', color=colors[name], linewidth=2, linestyle='--', marker='s', markersize=4)
    ax.set_title(f'{name} — RMSE: {metrics_results[name]["RMSE"]:.1f} | R²: {metrics_results[name]["R2"]:.3f}', fontsize=11)
    ax.set_ylabel('Passagers (milliers)')
    ax.legend(loc='upper left')

axes[-1].set_xlabel('Mois (jeu de test)')
plt.suptitle('Prédictions vs Réalité — Comparaison des 3 modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Graphique combiné : tous les modèles sur le même plot ───────────────────
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(y_test_real, label='Valeurs réelles', color='black', linewidth=2.5, marker='o', markersize=5, zorder=5)
for name, y_pred in predictions.items():
    ax.plot(y_pred, label=name, color=colors[name], linewidth=1.5, linestyle='--', alpha=0.85)

ax.set_title('Comparaison globale — Réel vs RNN vs LSTM vs GRU', fontsize=14, fontweight='bold')
ax.set_xlabel('Mois (jeu de test)')
ax.set_ylabel('Passagers (milliers)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparaison des métriques en barres ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metric_names = ['RMSE', 'MAE', 'R2']
metric_labels = ['RMSE (plus bas = mieux)', 'MAE (plus bas = mieux)', 'R² (plus haut = mieux)']

for ax, metric, label in zip(axes, metric_names, metric_labels):
    values = [metrics_results[name][metric] for name in models_dict.keys()]
    bars = ax.bar(models_dict.keys(), values,
                  color=[colors[n] for n in models_dict.keys()], edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.3f}' if metric == 'R2' else f'{v:.1f}',
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_title(label, fontsize=11)

plt.suptitle('Comparaison des métriques de performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Détermination du meilleur modèle ─────────────────────────────────────────
best_model_name = min(metrics_results, key=lambda n: metrics_results[n]['RMSE'])

print("🏆 RÉSUMÉ DES PERFORMANCES :\n")
print(f"{'Modèle':<12} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
print("-" * 44)
for name in models_dict.keys():
    m = metrics_results[name]
    marker = ' 🏆' if name == best_model_name else ''
    print(f"{name:<12} {m['RMSE']:>10.2f} {m['MAE']:>10.2f} {m['R2']:>10.4f}{marker}")

print(f"\n✅ Meilleur modèle : {best_model_name} (RMSE le plus faible)")

---
## 🎓 Conclusion

Vous avez construit, entraîné et comparé **3 architectures récurrentes** sur une tâche de prévision de série temporelle réelle :

| Partie | Ce qu'on a fait |
|--------|----------------|
| **1 - Données** | Chargé, nettoyé (dropna), normalisé (MinMaxScaler), créé des séquences (6 mois → 1 mois) |
| **2 - Modèles** | Construit SimpleRNN, LSTM, GRU (2 couches empilées + Dropout + Dense) |
| **3 - Entraînement** | Entraîné avec Early Stopping (patience=20) |
| **4 - Historique** | Comparé les courbes de loss train/validation |
| **5 - Prédictions** | Prédit sur le test, dé-normalisé pour revenir à l'échelle réelle |
| **6 - Résultats** | Visualisé et comparé RMSE, MAE, R² pour déterminer le meilleur modèle |

### 💡 Points clés à retenir
- Sur de petits datasets (144 mois), la différence entre RNN/LSTM/GRU peut être faible
- Le **LSTM/GRU** apportent généralement un avantage sur des séquences plus longues ou plus complexes
- La **saisonnalité annuelle** des données de passagers aériens est bien capturée avec `look_back=6` mois

### 🚀 Pour aller plus loin :
- Tester différentes valeurs de `LOOK_BACK` (3, 12, 24 mois)
- Ajouter des features supplémentaires (mois de l'année, tendance)
- Essayer un modèle **Bidirectionnel** ou un **Transformer** pour les séries temporelles